# Mejoras de trabajo final

A continuacion se detallaran las mejoras y correcciones propuestas en la correccion enviada por el profesor de la catedra:

---

## 1. MLOps y Código

Si bien en el trabajo final pudimos medir el RAG realizado en el MVP, es necesario seguir midiendo distintas funcionalidades de nuestro agente. A continuación planteamos cómo testeamos el uso de las herramientas del agente.

### 1.1 Tests automáticos

La suite quedó organizada por tema en tres archivos dentro de `tests/`:

- `tests/conftest.py`: solo configuración (opciones `--nmap-target`, `--nmap-arguments` y `--nmap-ports`, más sus fixtures). No contiene tests; existe porque pytest exige declarar las opciones de línea de comandos en un `conftest.py` o en un plugin.
- `tests/test_nmap.py`: 10 tests rápidos de seguridad/validación + 1 test de integración opt-in.
- `tests/test_pdf.py`: 5 tests de generación de PDFs.

Los tests se agrupan en dos temas:

#### Grupo 1: Tests de seguridad y validación de Nmap (`tests/test_nmap.py`)

Aquí realizamos dos tipos de tests:

* **Suite rápida (`tests/test_nmap.py`)**

Probamos únicamente la lógica de *hardening*, es decir, la allowlist, targets, puertos e inyecciones. No necesitamos pasarle una red real como parámetro, ya que su objetivo es comprobar que no ejecute comandos peligrosos, pero no chequea que los resultados del escaneo sean correctos. Está pensada para ejecutarse en MLOps/CI.

* **Test de red (`tests/test_nmap.py`, marker `network`)**

Testeamos que Nmap corra, encuentre hosts y devuelva el JSON esperado. Debemos pasarle una red en la cual pueda escanear de manera real a sus hosts. Es opt-in: sin `--nmap-target` se omite.

##### Tests disponibles

*15 funciones rápidas (50 casos):*

- **Allowlist de Nmap:** `--script`, inyecciones de shell, `-oN`/`-oX`, `-iL`, `-p`, spoofing (`-D`/`-S`/`-e`), exceso de argumentos y comillas sin cerrar.
- **Targets y puertos válidos/inválidos.**
- Invocación de la tool con args maliciosos sin ejecutar Nmap (scanner monkeypatcheado a una bomba).
- Camino feliz con un `PortScanner` falso.

*1 test de integración opcional:* escaneo real de la red que le pases, con resumen de hosts encontrados.

Lo ejecutamos de la siguiente manera:

```bash
# Suite rápida: el test de red se omite
uv run pytest
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Todos los tests de Nmap (rápidos + escaneo real; default -sn)
uv run pytest tests/test_nmap.py --nmap-target 192.168.0.0/24 -s
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Solo el test de red
uv run pytest -m network --nmap-target 192.168.0.0/24 -s
```


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 2.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Con flags permitidos y puertos
uv run pytest -m network --nmap-target 192.168.0.0/24 --nmap-arguments="-sT -F --open" --nmap-ports="22,80,443" -s
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 3.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Con variable de entorno
CAPAZETA_NMAP_TARGET=192.168.0.0/24 uv run pytest -m network -s
```


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 4.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

`-s` es para ver la lista de hosts; sin él, pytest captura el `print`.

Opciones:
- `--nmap-target` RED: IP, CIDR o hostname. Sin esto, el test se omite.
- `--nmap-arguments` ARGS: default `-sn`. Se validan contra la allowlist.
- `--nmap-ports` PUERTOS: default `1-1024`. Se ignoran con `-sn` y `-F`.

El target se valida con `_validate_target` y los flags con la misma allowlist que usa el agente, así que `--nmap-arguments "--script vuln"` se rechaza igual que desde la CLI. El test no genera reportes (`save_md`/`save_pdf` quedan en `False`).

#### Grupo 2: Tests de generación de PDFs (`tests/test_pdf.py`)

Los encontramos en [`tests/test_pdf.py`](tests/test_pdf.py).

| Test | Qué cubre |
|---|---|
| `test_markdown_a_pdf_genera_pdf_valido` | `markdown_to_pdf()` con título, encabezados, listas, negrita, itálica y código. Verifica que devuelve la ruta y genera un PDF real. |
| `test_pdf_con_tabla_enlaces_y_codigo` | Renderiza una tabla Markdown con enlaces, una cita y formato inline. Ejercita `_make_table`, `_split_row`, `_is_separator` y `_inline` sin excepciones. |
| `test_save_pdf_report_sanitiza_target` | `save_pdf_report()` con target `../../etc/passwd`. Comprueba que el archivo queda dentro del `reports_dir` indicado, con nombre `nmap_etc_passwd_<ts>.pdf`, sin `/` ni `..`, y que es un PDF válido. |
| `test_save_general_pdf_report_usa_prefix` | `save_general_pdf_report()` con prefijo `"informe wazuh"` → `informe_wazuh_<ts>.pdf`. |
| `test_pdf_con_contenido_vacio_o_html` (2 casos) | Contenido vacío → fallback `"Sin contenido."`; contenido con `<script>` y `&` → el `escape()` evita que ReportLab reviente al parsear el XML interno de `Paragraph`. |

`assert_pdf()` valida tres cosas concretas:
- el archivo existe,
- empieza con la firma `%PDF-`,
- termina en `%%EOF` y pesa más de 500 bytes.

*Qué no prueba*
- No compara el texto interno ni la cantidad de páginas (no se agregó pypdf ni dependencias extra).
- No valida la apariencia visual (márgenes, colores, fuentes).
- No prueba los wrappers de Wazuh/MITRE, ni `run_nmap_scan(save_pdf=True)` de punta a punta; prueba las funciones de `pdf_report.py` directamente.
- No escribe en `reports/`: todo va a `tmp_path`.

Probamos solo esos tests de la siguiente manera:

```bash
uv run pytest tests/test_pdf.py
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 5.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>


### Integracion continua

Vamos a subir el workflow de testing a Git Hub, asi se ejecutan los tests correspondientes cuando se realiza cada push. Vamos a testear solamente la integridad de comandos de nmap y el report de pdf, ya que para no podemos usar una network real al ejecutar la prueba en github, por lo que este quedara afuera

Se agrego el siguiente `test.yml` dentro de la carpeta de `.github/workflow`
```bash
name: Tests

on:
  push:
    branches:
      - master
      - "features/**"
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: tests-${{ github.ref }}
  cancel-in-progress: true

jobs:
  pytest:
    name: pytest (suite rápida)
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - name: Checkout
        uses: actions/checkout@v5

      - name: Instalar uv y Python 3.10
        uses: astral-sh/setup-uv@v10
        with:
          python-version: "3.10"
          enable-cache: true

      - name: Instalar dependencias
        run: uv sync --locked --group dev

      - name: Correr tests sin red
        run: uv run --no-sync pytest -m "not network"
```
Una vez realizado del commit y el push correspondientes vamos a git hub para ver si se ejecutaron los tests al pushear


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 6.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Vemos que se ejecutaron y superaron los tests correctamente

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 7.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

---

## 2. Armado de Ground Truth y evaluacion del RAG

### 2.1. Documentacion 

A pesar de que se documento correctamente la metodologia de clasificacion de etiquetas en el `notebook/02_evaluacion_rag.ipynb`, se redacto de manera mas detallada en el *README.md* del proyecto.

- En `README.md` → **Metodología → Ground truth de la evaluación RAG**:
  - Se aclara que la técnica de referencia sale del **mapeo MITRE declarado por las propias reglas de Wazuh** (`rule.id` → `mitre.id`), no de una comparación manual contra el catálogo.
  - Se documenta que el conjunto tiene **31 alertas** (descripciones) provenientes de **25 reglas únicas**; una misma regla puede aparecer en varias filas cuando Wazuh emite descripciones distintas (por ejemplo, la 80791 aparece con cuatro rutas borradas).
  - Se explica que la consulta al retriever es `rule.description` y que la columna `Count` no se usa para ponderar la métrica.
  - Se documentan los casos sin mapeo explícito: la regla 80783 (`Audit: Watch - Read access.`) se marcó como `Inferido` y se mapeó a T1003.008 por el watch de lectura sobre `/etc/shadow`.
  - Se listan las cinco filas con notas: 80783 (`Inferido`), 554 (alternativa T1565.001), 80781 (alternativa T1098), 521 (probable falso positivo) y 5301 (`Fallo de su`).
- En `README.md` → **Resultados**:
  - Se corrigió la definición de Hit@5 y MRR para dejar claro que se comparan contra el `mitre_id` principal anotado.
  - Se agregó un párrafo de limitaciones: el ground truth es de etiqueta única y las alternativas no se acreditan. En las corridas registradas esas alternativas no aparecieron en el Top 5, por lo que los números actuales no cambian. La fila inferida puede mover el Hit@5 como máximo 3.2 pp (1/31) y el MRR hasta 0.032.
- En el **Roadmap** se agregó como mejora futura el **ground truth multi-etiqueta**.

**Dónde está el procedimiento de clasificación:** la construcción del dataset, la consulta al retriever y el cálculo de Hit@5/MRR están en [`notebooks/02_evaluacion_rag.ipynb`](02_evaluacion_rag.ipynb); la comparación entre modelos de embeddings, en [`notebooks/03_evaluacion_modelos_embedding.ipynb`](03_evaluacion_modelos_embedding.ipynb). El CSV anotado es [`tests/wazuh_ground_truth_annoted.csv`](../tests/wazuh_ground_truth_annoted.csv) y el que consume LangSmith, [`tests/dataset_evaluacion_langsmith.csv`](../tests/dataset_evaluacion_langsmith.csv).

### 2.2. Intervalos de confianza y alcance del experimento

#### Qué se corrigió

La corrección pidió tres cosas: agregar intervalos de confianza del 95 % al Hit@5, aclarar que cada modelo se evaluó en una sola corrida y dejar asentado que, con 31 alertas y 17 técnicas distintas, los números son frágiles y que ampliar el dataset es la mejora prioritaria.

Se actualizaron:

- [`README.md`](../README.md) → **Resultados**: la tabla ahora tiene la columna **IC 95 % (Hit@5)** y un párrafo que explica el cálculo, la única corrida y la fragilidad muestral.
- [`README.md`](../README.md) → **Roadmap**: se agregó **Ampliación del dataset de evaluación**, con el compromiso de repetir la comparación y recalcular los IC con el dataset ampliado.
- [`docs/index.html`](../docs/index.html): misma tabla con IC y nota de alcance.
- [`notebooks/03_evaluacion_modelos_embedding.ipynb`](03_evaluacion_modelos_embedding.ipynb): se agregó el cálculo reproducible leyendo los CSV de `reports/` (sin re-ejecutar LangSmith) y la tabla comparativa con IC.
- [`notebooks/02_evaluacion_rag.ipynb`](02_evaluacion_rag.ipynb): se agregaron los IC de TF-IDF y Gemini, la fórmula y la aclaración de alcance. El procedimiento de clasificación de eventos está documentado en ese notebook.

#### Criterio de cálculo

Se usó la aproximación normal (Wald) para una proporción:

$$IC_{95\%} = \hat{p} \pm 1.96\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

donde $\hat{p}$ = aciertos / $n$ y $n$ = cantidad de alertas evaluadas (31 en todos los modelos reportados). Se eligió este criterio porque reproduce el rango indicado por el docente para Gemini, 52 %–84 %.

#### Resultados con IC

| Modelo | n | Aciertos | Hit@5 | IC 95 % |
| :--- | :---: | :---: | :---: | :---: |
| Gemini embedding-001 | 31 | 21 | 67.74 % | [51.29 %, 84.20 %] |
| Qwen3 embedding-0.6b | 31 | 17 | 54.84 % | [37.32 %, 72.36 %] |
| BGE-Large-EN-v1.5 | 31 | 16 | 51.61 % | [34.02 %, 69.20 %] |
| Baseline (TF-IDF) | 31 | 10 | 32.26 % | [15.80 %, 48.71 %] |

#### Lectura

Cada modelo se evaluó en **una sola corrida**. El dataset tiene **31 alertas, 25 reglas únicas y 17 técnicas MITRE distintas**. El IC de Gemini ([51.29 %, 84.20 %]) se superpone con los de Qwen3 ([37.32 %, 72.36 %]) y BGE ([34.02 %, 69.20 %]): la ventaja observada es indicativa, no concluyente. La diferencia frente a TF-IDF sí es clara porque los intervalos no se solapan.

#### Mejora necesaria

Ampliar el dataset es la mejora prioritaria. Se realizara un notebook especifico para ampliar el dataset y tener una mayor cantida de clasificacion de alertas.

**Aclaración:** no se modificaron datos, reportes ni resultados; solo documentación y el cálculo de los IC a partir de los CSV existentes en `reports/`.


### 2.3. Comparacion con el modelo Plamo

Al ser un modelo orientado al japones, Plamo fallo considerablemente en muchos de las clasificaciones, funcionando correctamente casi unicamente en la mitad, por lo que seria incorrecto medirlo con la misma vara que los otros modelos. Se opto por extraer toda referencia a este modelo en la etapa de evaluacion de que modelo de embedding conviene utilizar, dejando solamente a los otros dos competidores.

### 2.4. Tecnicas MITRE deprecadas

Al descargar el archivo de tecnicas MITRE correspondiente no se estaban filtrando las tecnicas las cuales fueron deprecadas, esto afecta solamente al TD-IDF, ya que 
el código existente en `src/capazeta/rag/mitre.py ya manejaba` este filtrado al construir el índice FAISS:
- La función _parse_techniques marca deprecated=True para técnicas revocadas
- La función _dedupe_by_id prefiere la versión vigente
- En build_index, solo se incluyen técnicas no deprecated: current = [t for t in by_id.values() if not t["deprecated"]]

Corremos el script de la siguiente manera
```bash
# Ver informe general
python filter_mitre_techniques.py --json data/mitre/enterprise-attack.json

# Filtrar IDs específicos
python filter_mitre_techniques.py --json data/mitre/enterprise-attack.json --techs T1062 T1108 T1131 T1001

# Generar JSON con solo técnicas vigentes (para usar en el índice)
python filter_mitre_techniques.py --json data/mitre/enterprise-attack.json --output techniques_vigentes.json
```

Ahora para ser mas certeros nuestro deber es evaluar nuevamente el TD-IDF para poder compararlos correctamente:

Primero armamos el dataset filtrado

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 8.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Copio el archivo en un lugar mas clean
```bash
(final) joaquincernik@dell:~/Desktop/ia/final$ cp techniques_vigentes.json data/mitre/
```

In [1]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargaoms la base de conocimiento de MITRE desde el JSON
json_path = "../data/mitre/techniques_vigentes.json" #cambiamos al filtrado

with open(json_path, 'r', encoding='utf-8') as f:
    mitre_data = json.load(f)

ids_mitre = []
textos_mitre = []

# Procesamiento exacto para  estructura de JSON
for t_id, info in mitre_data.items():
    ids_mitre.append(t_id)
    nombre = info.get("name", "")
    descripcion = info.get("description", "")
    textos_mitre.append(f"{nombre} {descripcion}")

print(f"Se cargaron {len(ids_mitre)} técnicas de MITRE exitosamente.")
# 2. Inicializamos y ajustamos el modelo TF-IDF (El motor de palabras clave)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(textos_mitre)

# 3. Definimos el pipeline del Baseline para LangSmith
def keyword_baseline_pipeline(inputs: dict) -> dict:
    """
    Busca las 5 técnicas de MITRE más similares a la alerta usando coincidencia exacta de palabras (TF-IDF).
    """
    alerta_texto = inputs["rule.description"]
    
    # Vectorizamos la alerta entrante
    alerta_vec = vectorizer.transform([alerta_texto])
    
    # Calculamos la similitud contra todas las técnicas de MITRE
    similitudes = cosine_similarity(alerta_vec, tfidf_matrix).flatten()
    
    # Obtenemos los índices de los 5 resultados con mayor puntuación
    top_5_indices = similitudes.argsort()[-5:][::-1]
    
    # Extraemos los IDs correspondientes
    tecnicas_recuperadas = [ids_mitre[i] for i in top_5_indices]
    
    return {"tecnicas_recuperadas": tecnicas_recuperadas}

# Prueba rápida manual antes de ejecutar LangSmith:
ejemplo_alerta = {"rule.description": "Successful sudo to ROOT executed"}
print("Predicción Baseline:", keyword_baseline_pipeline(ejemplo_alerta))

Se cargaron 706 técnicas de MITRE exitosamente.
Predicción Baseline: {'tecnicas_recuperadas': ['T1548.003', 'T1553.004', 'T1098.007', 'T1078.001', 'T1548.004']}


In [2]:
from langsmith.evaluation import evaluate
def mitre_retrieval_evaluator(run, example) -> list[dict]:
    """
    Evaluador robusto a prueba de errores de parseo o tipos de datos en LangSmith.
    """
    try:
        # 1. Extraer la técnica esperada de forma segura
        ref_out = example.outputs.get("mitre_id") or example.outputs.get("reference.mitre_id")
        if isinstance(ref_out, dict):
            tecnica_esperada = str(ref_out.get("mitre_id", "")).strip()
        else:
            tecnica_esperada = str(ref_out).strip()
            
        # 2. Extraer las técnicas recuperadas por la corrida
        outputs = run.outputs or {}
        tecnicas_recuperadas = outputs.get("tecnicas_recuperadas", [])
        
        # Sanitizar lista a strings simples
        tecnicas_recuperadas = [str(t).strip() for t in tecnicas_recuperadas]
        
        # 3. Calcular Hit@5
        hit = 1.0 if tecnica_esperada in tecnicas_recuperadas else 0.0
        
        # 4. Calcular MRR
        if tecnica_esperada in tecnicas_recuperadas:
            rank = tecnicas_recuperadas.index(tecnica_esperada) + 1
            mrr = 1.0 / rank
        else:
            mrr = 0.0

        return [
            {"key": "hit@5", "score": hit},
            {"key": "mrr", "score": mrr}
        ]

    except Exception as e:
        # Si ocurre cualquier falla inesperada, devuelve 0.0 en lugar de dejar la fila vacía (No feedback)
        return [
            {"key": "hit@5", "score": 0.0},
            {"key": "mrr", "score": 0.0}
        ]
    
# 2. Ejecutar la Evaluación en LangSmith
dataset_name = "Evaluacion_Wazuh_MITRE"


### Re-evaluacion de TD-IDF

In [3]:

print("Iniciando evaluación del Baseline por Palabras Clave (TF-IDF)...")

results_baseline = evaluate(
    keyword_baseline_pipeline,
    data=dataset_name,
    evaluators=[mitre_retrieval_evaluator],
    experiment_prefix="Baseline_Keywords_TFIDF",
)
df_tf_idf = results_baseline.to_pandas()
df_tf_idf.to_csv(f"../reports/evaluacion_capazeta_tf_idf.csv", index=False)
print("¡Evaluación finalizada! Revisa tu panel de LangSmith.")

Iniciando evaluación del Baseline por Palabras Clave (TF-IDF)...
View the evaluation results for experiment: 'Baseline_Keywords_TFIDF-738b19a4' at:
https://smith.langchain.com/o/954863b9-6bd1-4509-9100-6980d0c6d6e0/datasets/2415f52e-806c-4939-b514-adada324b15c/compare?selectedSessions=2bc14f9a-0f6d-4352-97ba-42c846b69e0a


¡Evaluación finalizada! Revisa tu panel de LangSmith.


Observemos el resultado

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 9.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Si analizamos el csv en `reports/evaluacion_capazeta_tf_idf.csv`, vemos que no hay ninguna tecnica deprecada.

Vamos a compararlo con la evaluacion anterior la cual contemplaba tecnicas deprecadas como validas

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 10.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

De ahi podemos extraer la siguiente tabla

 Métrica | Modelo No Filtrado (`Baseline Keywords TFIDF`) | Modelo Filtrado (`...-738b19a4`) | Variación |
| :--- | :--- | :--- | :--- |
| **Aciertos (Hit@5)** | 32.26 % | 32.26 % | **Sin cambios** |
| **Ranking (MRR)** | 27.69 % | 22.58 % | **Cayó un 18.4%** |
| **Latencia Promedio** | 0.0068 segundos | 0.0026 segundos | **Mejoró un 61.7%** |

### Conclusiones de la diferencia

* **Capacidad de recuperación intacta (Hit@5):** El filtrado no afectó en absoluto la capacidad del modelo para incluir la técnica correcta dentro del Top 5 de resultados. Ambos modelos fallan y aciertan exactamente en los mismos casos base.
* **Degradación en el ordenamiento (MRR):** Aunque el modelo filtrado sigue encontrando la técnica en el Top 5, la posiciona más abajo en la lista. El modelo original sin filtrar era mejor colocando la respuesta correcta en el primer o segundo lugar. El proceso de filtrado eliminó palabras clave o características que ayudaban a desempatar y priorizar correctamente la técnica exacta.
* **Optimización de velocidad (Latencia):** La principal ventaja del filtrado es el rendimiento computacional. Al reducir el espacio de búsqueda (ya sea eliminando técnicas deprecadas, reduciendo el vocabulario o filtrando *stopwords*), la matriz matemática se volvió más ligera, permitiendo que la búsqueda vectorial se ejecute casi tres veces más rápido.


---

## 3. Evaluacion del LLM

En base a las observaciones del profe, se considero necesario dedicarle un notebook espeifico a la evaluacion del LLM, visite el notebook 07